# DDSM Overlay — Visualizador de Lesiones

Notebook de demostración del módulo `src/ddsm_overlay.py`.  
Permite seleccionar una imagen mamográfica del dataset **6 DDSM** y visualizar el contorno de la lesión reconstruido desde el chain code Freeman del archivo `.OVERLAY`.

**Flujo:**
1. Seleccionar categoría (benigno / cáncer)
2. Seleccionar grupo y caso
3. Seleccionar la vista mamográfica disponible
4. Ver la imagen con el contorno superpuesto, el ROI recortado y los metadatos clínicos

In [1]:
import sys
from pathlib import Path

# Agregar src/ al path para importar ddsm_overlay
TESIS_ROOT = Path(".").resolve()
sys.path.insert(0, str(TESIS_ROOT / "src"))

DDSM_ROOT = TESIS_ROOT / "data" / "6 DDSM"
print(f"Tesis root : {TESIS_ROOT}")
print(f"DDSM root  : {DDSM_ROOT}")
print(f"Existe     : {DDSM_ROOT.exists()}")

Tesis root : /home/gtrujillod/Tesis
DDSM root  : /home/gtrujillod/Tesis/data/6 DDSM
Existe     : True


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

from ddsm_overlay import (
    parse_overlay,
    draw_lesion_contour,
    crop_lesion_roi,
    create_lesion_mask,
    find_image_for_overlay,
)

print("Imports OK")

Imports OK


## Indexar los archivos OVERLAY disponibles

In [3]:
# Construir índice: overlay_path → metadata mínima (categoría, grupo, caso, vista)
def build_index(ddsm_root: Path) -> dict:
    """
    Escanea todos los .OVERLAY y devuelve un dict anidado:
        index[categoria][grupo][caso] = [overlay_path, ...]
    """
    index = {}
    for ov in sorted(ddsm_root.rglob("*.OVERLAY")):
        parts = ov.parts
        # Categoría: 'benign cases' o 'cancer cases'
        cat = next((p for p in parts if "cases" in p.lower()), "unknown")
        # Grupo: benign_XX / cancer_XX  (padre del case)
        grupo = ov.parent.parent.name   # ej. cancer_04
        caso  = ov.parent.name          # ej. case1081
        index.setdefault(cat, {}).setdefault(grupo, {}).setdefault(caso, []).append(ov)
    return index

INDEX = build_index(DDSM_ROOT)
categorias = sorted(INDEX.keys())
print(f"Categorías encontradas: {categorias}")
for cat, grupos in INDEX.items():
    total_casos = sum(len(casos) for casos in grupos.values())
    total_ovs   = sum(len(ovs) for casos in grupos.values() for ovs in casos.values())
    print(f"  {cat}: {len(grupos)} grupos, {total_casos} casos, {total_ovs} overlays")

Categorías encontradas: ['benign cases', 'cancer cases']
  benign cases: 14 grupos, 855 casos, 1774 overlays
  cancer cases: 15 grupos, 913 casos, 1897 overlays


## Función de visualización

In [4]:
def render_overlay(overlay_path: Path, zoom_pad: int = 300, figsize=(20, 8)):
    """
    Carga la imagen ORIGINAL (16-bit, sin ninguna transformación) y dibuja
    el contorno de la lesión reconstruido desde el chain code del .OVERLAY.

    Tres paneles:
    ① Mamografía completa con bbox de la lesión
    ② Zoom sobre la región de la lesión + contorno como plot()
    ③ Forma aislada del contorno (coordenadas relativas al bbox)
    """
    ov_data = parse_overlay(overlay_path)

    if ov_data.image_path is None or not ov_data.image_path.exists():
        print(f"⚠  Imagen no encontrada para: {overlay_path.name}")
        return

    # ── Cargar imagen 16-bit SIN NINGUNA TRANSFORMACIÓN ─────────────────
    # PIL abre los PNG del DDSM en modo I;16 (16-bit grayscale, uint16).
    # np.array() entrega directamente el array uint16 con los valores
    # originales del scanner (0-65535). Matplotlib muestra uint16 con
    # auto-escala al rango real de datos, sin recortar nada.
    arr = np.array(Image.open(ov_data.image_path))   # uint16, sin convert()
    img_h, img_w = arr.shape

    all_contours = ov_data.all_contours
    if not all_contours:
        print("⚠  No hay contornos reconstruibles en este overlay.")
        return

    # Bounding box global de todas las lesiones + padding para el zoom
    gx0 = max(0,     min(c.bbox[0] for c in all_contours) - zoom_pad)
    gy0 = max(0,     min(c.bbox[1] for c in all_contours) - zoom_pad)
    gx1 = min(img_w, max(c.bbox[2] for c in all_contours) + zoom_pad)
    gy1 = min(img_h, max(c.bbox[3] for c in all_contours) + zoom_pad)

    zoom_arr = arr[gy0:gy1, gx0:gx1]

    n_les  = len(ov_data.lesions)
    widths = [1.8, 3.0] + [1.2] * n_les
    fig, axes = plt.subplots(1, 2 + n_les, figsize=figsize,
                             gridspec_kw={"width_ratios": widths})
    if (2 + n_les) == 1:
        axes = [axes]

    COL = {True: "#FF3B30", False: "#30D158"}   # rojo=maligno · verde=benigno

    # ── ① Mamografía completa ───────────────────────────────────────────
    ax_full = axes[0]
    ax_full.imshow(arr, cmap="gray", aspect="auto")   # matplotlib auto-escala uint16
    for les in ov_data.lesions:
        col = COL[les.is_malignant]
        for c in les.contours:
            bx = c.bbox
            ax_full.add_patch(plt.Rectangle(
                (bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1],
                linewidth=2, edgecolor=col, facecolor="none"
            ))
    ax_full.set_title(f"Mamografía completa\n{img_w}×{img_h} px", fontsize=8)
    ax_full.axis("off")

    # ── ② Zoom + contorno ───────────────────────────────────────────────
    ax_zoom = axes[1]
    ax_zoom.imshow(zoom_arr, cmap="gray",
                   extent=[gx0, gx1, gy1, gy0],   # mantiene coordenadas originales
                   aspect="auto")
    for les in ov_data.lesions:
        col = COL[les.is_malignant]
        for c in les.contours:
            xs = np.append(c.x, c.x[0])
            ys = np.append(c.y, c.y[0])
            ax_zoom.plot(xs, ys, color="white", linewidth=4, alpha=0.55, zorder=2)
            ax_zoom.plot(xs, ys, color=col,    linewidth=1.8, zorder=3)
            ax_zoom.plot(c.x[0], c.y[0], "o", color=col, markersize=5,
                         markeredgecolor="white", markeredgewidth=1.2, zorder=4)
    ax_zoom.set_xlim(gx0, gx1)
    ax_zoom.set_ylim(gy1, gy0)
    view_tag = overlay_path.stem.split(".")[1] if "." in overlay_path.stem else ""
    ax_zoom.set_title(
        f"Zoom lesión  ·  vista {view_tag}\n"
        f"imagen original 16-bit  (pad={zoom_pad} px)",
        fontsize=8
    )
    ax_zoom.axis("off")

    # ── ③ Forma aislada del contorno ────────────────────────────────────
    for i, les in enumerate(ov_data.lesions):
        ax_s = axes[2 + i]
        col  = COL[les.is_malignant]
        for c in les.contours:
            xs_r = c.x - c.bbox[0]
            ys_r = c.y - c.bbox[1]
            ax_s.fill(xs_r, ys_r, color=col, alpha=0.2)
            ax_s.plot(np.append(xs_r, xs_r[0]),
                      np.append(ys_r, ys_r[0]),
                      color=col, linewidth=1.5)
        lbl = "MAL" if les.is_malignant else "BEN"
        ax_s.set_title(f"Contorno {les.abnormality_id}\n{les.lesion_type} · {lbl}", fontsize=8)
        ax_s.set_aspect("equal")
        ax_s.invert_yaxis()
        ax_s.axis("off")

    plt.suptitle(
        f"{overlay_path.parent.name}  ·  {overlay_path.name}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

    # ── Metadatos clínicos ──────────────────────────────────────────────
    print("─" * 65)
    print(f"  Overlay : {overlay_path.name}")
    print(f"  Imagen  : {ov_data.image_path.name}  ({img_w}×{img_h} px, 16-bit)")
    print(f"  Rango   : {arr.min()} – {arr.max()}")
    print("─" * 65)
    for les in ov_data.lesions:
        tag = "🔴 MALIGNO" if les.is_malignant else "🟢 BENIGNO"
        print(f"  Anomalía #{les.abnormality_id}  {tag}")
        print(f"    BI-RADS    : {les.assessment}")
        print(f"    Subtlety   : {les.subtlety}  (1=fácil · 5=difícil)")
        print(f"    Tipo       : {les.lesion_type}")
        if les.lesion_type == "MASS":
            print(f"    Forma      : {les.mass_shape}")
            print(f"    Márgenes   : {les.mass_margins}")
        elif les.lesion_type == "CALCIFICATION":
            print(f"    Tipo calc. : {les.calc_type}")
            print(f"    Distribuc. : {les.calc_distribution}")
        for c in les.contours:
            bx = c.bbox
            print(f"    Contorno {c.outline_id} : {len(c.x)} pts  "
                  f"({bx[0]},{bx[1]})→({bx[2]},{bx[3]})  "
                  f"{bx[2]-bx[0]}×{bx[3]-bx[1]} px")
    print("─" * 65)

print("Función render_overlay definida ✓")

Función render_overlay definida ✓


## Selector interactivo

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────────
w_cat = widgets.Dropdown(
    options=categorias,
    description="Categoría:",
    layout=widgets.Layout(width="340px"),
    style={"description_width": "90px"},
)
w_grupo = widgets.Dropdown(
    description="Grupo:",
    layout=widgets.Layout(width="340px"),
    style={"description_width": "90px"},
)
w_caso = widgets.Dropdown(
    description="Caso:",
    layout=widgets.Layout(width="340px"),
    style={"description_width": "90px"},
)
w_vista = widgets.Dropdown(
    description="Vista:",
    layout=widgets.Layout(width="340px"),
    style={"description_width": "90px"},
)
w_btn = widgets.Button(
    description="Mostrar lesión",
    button_style="primary",
    icon="eye",
    layout=widgets.Layout(width="200px", height="36px"),
)
w_btn_random = widgets.Button(
    description="Caso aleatorio",
    button_style="info",
    icon="random",
    layout=widgets.Layout(width="200px", height="36px"),
)
w_out = widgets.Output()

# ── Lógica de cascada ────────────────────────────────────────────────────
def update_grupos(change):
    cat = w_cat.value
    grupos = sorted(INDEX.get(cat, {}).keys())
    w_grupo.options = grupos
    if grupos:
        w_grupo.value = grupos[0]

def update_casos(change):
    cat   = w_cat.value
    grupo = w_grupo.value
    casos = sorted(INDEX.get(cat, {}).get(grupo, {}).keys())
    w_caso.options = casos
    if casos:
        w_caso.value = casos[0]

def update_vistas(change):
    cat   = w_cat.value
    grupo = w_grupo.value
    caso  = w_caso.value
    ovs   = INDEX.get(cat, {}).get(grupo, {}).get(caso, [])
    # Mostrar solo el nombre del .OVERLAY como etiqueta
    opciones = [(ov.name, ov) for ov in sorted(ovs)]
    w_vista.options = opciones
    if opciones:
        w_vista.value = opciones[0][1]

w_cat.observe(update_grupos,  names="value")
w_grupo.observe(update_casos, names="value")
w_caso.observe(update_vistas, names="value")

# Inicializar en cascada
update_grupos(None)
update_casos(None)
update_vistas(None)

# ── Botón Mostrar ────────────────────────────────────────────────────────
def on_mostrar(b):
    with w_out:
        clear_output(wait=True)
        ov_path = w_vista.value
        if ov_path is None:
            print("Selecciona un overlay primero.")
            return
        render_overlay(Path(ov_path))

# ── Botón Aleatorio ──────────────────────────────────────────────────────
import random

def on_random(b):
    cat   = random.choice(list(INDEX.keys()))
    grupo = random.choice(list(INDEX[cat].keys()))
    caso  = random.choice(list(INDEX[cat][grupo].keys()))
    w_cat.value   = cat
    w_grupo.value = grupo
    w_caso.value  = caso
    on_mostrar(None)

w_btn.on_click(on_mostrar)
w_btn_random.on_click(on_random)

# ── Layout final ─────────────────────────────────────────────────────────
panel = widgets.VBox([
    widgets.HTML("<b>Seleccionar imagen mamográfica DDSM</b>"),
    widgets.HBox([w_cat, w_grupo]),
    widgets.HBox([w_caso, w_vista]),
    widgets.HBox([w_btn, w_btn_random]),
])

display(panel, w_out)

Output()

---
## Uso programático (sin widgets)

Si prefieres especificar directamente un path de overlay:

In [ ]:
# Ejemplo directo — edita el path y ejecuta esta celda
overlay_manual = DDSM_ROOT / "cancer cases" / "cancers" / "cancer_04" / "case1081" / "A_1081_1.RIGHT_CC.OVERLAY"
render_overlay(overlay_manual)